In [ ]:
from io import StringIO

import pandas as pd
import requests

def requestConstituents(url, table_id):
    response = requests.get(
        url,
        headers={"User-Agent": "AAIS student project"},
        timeout=30,
    )
    response.raise_for_status()

    tables = pd.read_html(
        StringIO(response.text),
        attrs={"id": table_id},
        flavor="lxml",
    )
    df = tables[0]
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [
            "_".join(dict.fromkeys(str(level) for level in column if level))
            for column in df.columns
        ]
    df.columns = df.columns.str.lower().str.replace(" ", "_")
 
    
    return df

constituents_df = (
    requestConstituents(
        "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies",
        "constituents",
    )
)
history_df = (
    requestConstituents(
        "https://en.wikipedia.org/wiki/Historical_components_of_the_S&P_500",
          "changes",
          )
)
constituents_df["yahoo_symbol"] = constituents_df["symbol"].str.replace(".", "-", regex=False)

history_df['yahoo_added_ticker'] = history_df['added_ticker'].str.replace(".", "-", regex=False)
history_df['yahoo_removed_ticker'] = history_df['removed_ticker'].str.replace(".", "-", regex=False)


,symbol,security,gics_sector,gics_sub-industry,headquarters_location,date_added,cik,founded,yahoo_symbol
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,66740,1902,MMM
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,91142,1916,AOS
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800,1888,ABT
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,1551152,2013 (1888),ABBV
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,1467373,1989,ACN
...,...,...,...,...,...,...,...,...,...
498,XYL,Xylem Inc.,Industrials,Industrial Machinery & Supplies & Components,"White Plains, New York",2011-11-01,1524472,2011,XYL
499,YUM,Yum! Brands,Consumer Discretionary,Restaurants,"Louisville, Kentucky",1997-10-06,1041061,1997,YUM
500,ZBRA,Zebra Technologies,Information Technology,Electronic Equipment & Instruments,"Lincolnshire, Illinois",2019-12-23,877212,1969,ZBRA
501,ZBH,Zimmer Biomet,Health Care,Health Care Equipment,"Warsaw, Indiana",2001-08-07,1136869,1927,ZBH


In [121]:
constituents_df[['symbol', 'yahoo_symbol', 'date_added']][constituents_df['symbol'] == 'BE']

,symbol,yahoo_symbol,date_added
67,BE,BE,2026-09-21


In [126]:
def prepare_events_df(history_df):
    df = history_df.copy()
    df["date"] = pd.to_datetime(df["effective_date"], format="%B %d, %Y", errors="coerce")

    added = df[["date", "added_ticker"]].rename(columns={"added_ticker": "ticker"}).assign(action="add")
    removed = df[["date", "removed_ticker"]].rename(columns={"removed_ticker": "ticker"}).assign(action="remove")

    events_df = pd.concat([added, removed], ignore_index=True)
    events_df["ticker"] = events_df["ticker"].astype("string").str.strip()
    events_df = events_df.dropna(subset=["date", "ticker"])
    events_df = events_df[events_df["ticker"] != ""]

    return events_df.sort_values("date").reset_index(drop=True)



def members_on(intervals, date):
    d = pd.Timestamp(date)
    started = intervals["start"].isna() | (intervals["start"] <= d)
    not_ended = intervals["end"].isna() | (intervals["end"] > d)
    return sorted(intervals.loc[started & not_ended, "ticker"])



In [123]:
events_df = prepare_events_df(history_df)
intervals = []

for event in events_df.sort_values("date").itertuples():
    if event.action == "add":
        intervals.append((event.ticker, event.date, None))
    else:
        for i in range(len(intervals)):
            if intervals[i][0] == event.ticker and intervals[i][2] is None:
                intervals[i] = (intervals[i][0], intervals[i][1], event.date)
                break
        else:
            intervals.append((event.ticker, None, event.date))

constituents_df["date_added"] = pd.to_datetime(constituents_df["date_added"], errors="coerce")

open_tickers = {t for t, _, end in intervals if end is None}
for row in constituents_df.itertuples():
    if row.yahoo_symbol not in open_tickers:
        intervals.append((row.yahoo_symbol, row.date_added, None))

In [124]:
iv = pd.DataFrame(intervals, columns=["ticker", "start", "end"])
phantoms = [
    "ACT", "ALLE |", "BUD", "CCR", "COG", "DLPH", "FB", "FLT", "FSR", "HRS", "JEC",
    "JOYG", "KORS", "LUK", "NCC", "PCLN", "RE", "SATS", "TSO", "UA", "WLP",
]



iv = iv[~iv["ticker"].isin(phantoms)]

iv = iv[~((iv["ticker"] == "IR") & iv["end"].isna() & ~(iv["start"] >= "2020-01-01"))]
iv = iv.reset_index(drop=True)
print(iv["ticker"].nunique(), len(iv))
print(iv.loc[iv["end"].isna(), "ticker"].nunique(), len(constituents_df))
print(iv[iv["start"] > iv["end"]])
print(iv[iv["end"].isna()].groupby("ticker").size().loc[lambda s: s > 1])

for d in ["2012-01-03", "2016-01-04", "2020-01-02", "2024-01-02"]:
    print(d, len(members_on(iv, d)))

860 890
503 503
Empty DataFrame
Columns: [ticker, start, end]
Index: []
Series([], dtype: int64)
2012-01-03 496
2016-01-04 500
2020-01-02 503
2024-01-02 502
